<a href="https://colab.research.google.com/github/hamshini1413/NLP/blob/main/NLP7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:


!pip install nltk wordcloud textblob

In [2]:
!pip install -q kaggle
from google.colab import files
files.upload()
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d jiashenliu/515k-hotel-reviews-data-in-europe
!unzip -o 515k-hotel-reviews-data-in-europe.zip

cp: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory
Dataset URL: https://www.kaggle.com/datasets/jiashenliu/515k-hotel-reviews-data-in-europe
License(s): CC0-1.0
100% 45.1M/45.1M [00:00<00:00, 154MB/s]

Archive:  515k-hotel-reviews-data-in-europe.zip
  inflating: Hotel_Reviews.csv       


In [3]:
import pandas as pd
import numpy as np
import nltk
import string

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from textblob import TextBlob

In [4]:
nltk.download("stopwords")
df = pd.read_csv("Hotel_Reviews.csv")

df.head()

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


,Hotel_Address,Additional_Number_of_Scoring,Review_Date,Average_Score,Hotel_Name,Reviewer_Nationality,Negative_Review,Review_Total_Negative_Word_Counts,Total_Number_of_Reviews,Positive_Review,Review_Total_Positive_Word_Counts,Total_Number_of_Reviews_Reviewer_Has_Given,Reviewer_Score,Tags,days_since_review,lat,lng
0,s Gravesandestraat 55 Oost 1092 AA Amsterdam ...,194,8/3/2017,7.7,Hotel Arena,Russia,I am so angry that i made this post available...,397,1403,Only the park outside of the hotel was beauti...,11,7,2.9,"[' Leisure trip ', ' Couple ', ' Duplex Double...",0 days,52.360576,4.915968
1,s Gravesandestraat 55 Oost 1092 AA Amsterdam ...,194,8/3/2017,7.7,Hotel Arena,Ireland,No Negative,0,1403,No real complaints the hotel was great great ...,105,7,7.5,"[' Leisure trip ', ' Couple ', ' Duplex Double...",0 days,52.360576,4.915968
2,s Gravesandestraat 55 Oost 1092 AA Amsterdam ...,194,7/31/2017,7.7,Hotel Arena,Australia,Rooms are nice but for elderly a bit difficul...,42,1403,Location was good and staff were ok It is cut...,21,9,7.1,"[' Leisure trip ', ' Family with young childre...",3 days,52.360576,4.915968
3,s Gravesandestraat 55 Oost 1092 AA Amsterdam ...,194,7/31/2017,7.7,Hotel Arena,United Kingdom,My room was dirty and I was afraid to walk ba...,210,1403,Great location in nice surroundings the bar a...,26,1,3.8,"[' Leisure trip ', ' Solo traveler ', ' Duplex...",3 days,52.360576,4.915968
4,s Gravesandestraat 55 Oost 1092 AA Amsterdam ...,194,7/24/2017,7.7,Hotel Arena,New Zealand,You When I booked with your company on line y...,140,1403,Amazing location and building Romantic setting,8,3,6.7,"[' Leisure trip ', ' Couple ', ' Suite ', ' St...",10 days,52.360576,4.915968


In [5]:
df["Review"] = df["Positive_Review"] + " " + df["Negative_Review"]

df = df[["Hotel_Name","Reviewer_Score","Review"]]

df.head()

,Hotel_Name,Reviewer_Score,Review
0,Hotel Arena,2.9,Only the park outside of the hotel was beauti...
1,Hotel Arena,7.5,No real complaints the hotel was great great ...
2,Hotel Arena,7.1,Location was good and staff were ok It is cut...
3,Hotel Arena,3.8,Great location in nice surroundings the bar a...
4,Hotel Arena,6.7,Amazing location and building Romantic settin...


In [6]:
stemmer = PorterStemmer()

stop_words = set(stopwords.words("english"))

def clean_text(text):

    text = str(text).lower()

    text = "".join(c for c in text if c not in string.punctuation)

    words = text.split()

    words = [word for word in words if word not in stop_words]

    words = [stemmer.stem(word) for word in words]

    return " ".join(words)

In [10]:
df.columns

Index(['Hotel_Name', 'Reviewer_Score', 'Review'], dtype='object')

In [16]:
df["Clean_Review"] = df["Review"].apply(clean_text)

df.head()

,Hotel_Name,Reviewer_Score,Review,Clean_score,Clean_Review
0,Hotel Arena,2.9,Only the park outside of the hotel was beauti...,park outsid hotel beauti angri made post avail...,park outsid hotel beauti angri made post avail...
1,Hotel Arena,7.5,No real complaints the hotel was great great ...,real complaint hotel great great locat surroun...,real complaint hotel great great locat surroun...
2,Hotel Arena,7.1,Location was good and staff were ok It is cut...,locat good staff ok cute hotel breakfast rang ...,locat good staff ok cute hotel breakfast rang ...
3,Hotel Arena,3.8,Great location in nice surroundings the bar a...,great locat nice surround bar restaur nice lov...,great locat nice surround bar restaur nice lov...
4,Hotel Arena,6.7,Amazing location and building Romantic settin...,amaz locat build romant set book compani line ...,amaz locat build romant set book compani line ...


In [17]:
def sentiment(review):

    score = TextBlob(review).sentiment.polarity

    if score > 0:
        return "Positive"

    elif score < 0:
        return "Negative"

    else:
        return "Neutral"

df["Sentiment"] = df["Clean_Review"].apply(sentiment)

df.head()

,Hotel_Name,Reviewer_Score,Review,Clean_score,Clean_Review,Sentiment
0,Hotel Arena,2.9,Only the park outside of the hotel was beauti...,park outsid hotel beauti angri made post avail...,park outsid hotel beauti angri made post avail...,Positive
1,Hotel Arena,7.5,No real complaints the hotel was great great ...,real complaint hotel great great locat surroun...,real complaint hotel great great locat surroun...,Positive
2,Hotel Arena,7.1,Location was good and staff were ok It is cut...,locat good staff ok cute hotel breakfast rang ...,locat good staff ok cute hotel breakfast rang ...,Positive
3,Hotel Arena,3.8,Great location in nice surroundings the bar a...,great locat nice surround bar restaur nice lov...,great locat nice surround bar restaur nice lov...,Positive
4,Hotel Arena,6.7,Amazing location and building Romantic settin...,amaz locat build romant set book compani line ...,amaz locat build romant set book compani line ...,Positive


In [18]:
df["Sentiment"].value_counts()

,count
Sentiment,
Positive,362137
Neutral,101623
Negative,51978


In [19]:
average_rating = df["Reviewer_Score"].mean()

print("Average Customer Rating:", average_rating)

Average Customer Rating: 8.395076569886262


In [20]:
negative_reviews = df[df["Sentiment"]=="Negative"]

negative_reviews[["Hotel_Name","Review"]].head(10)

,Hotel_Name,Review
11,Hotel Arena,Style location rooms 6 30 AM started big noi...
37,Hotel Arena,Very nice hotel located in a park Ca 30 minut...
39,Hotel Arena,Location on the park with easy access to tram...
51,Hotel Arena,The location and views When arriving I was ...
60,Hotel Arena,The property is beautiful The place is compl...
76,Hotel Arena,The location on the park is excellent and ver...
90,Hotel Arena,Large bed Even allowing for the hotel being...
91,Hotel Arena,It s a beautiful hotel or should I say will b...
93,Hotel Arena,Anyways the staff was extremely friendly no c...
98,Hotel Arena,No Positive Got charged 50 for a birthday pac...


In [21]:
from collections import Counter

words = " ".join(negative_reviews["Clean_Review"])

word_list = words.split()

common = Counter(word_list)

common.most_common(20)

[('room', 46627),
 ('staff', 19673),
 ('small', 19171),
 ('hotel', 18068),
 ('locat', 17632),
 ('breakfast', 12362),
 ('bed', 11453),
 ('posit', 11089),
 ('poor', 8287),
 ('bad', 6435),
 ('help', 6320),
 ('friendli', 6029),
 ('bathroom', 5997),
 ('servic', 5931),
 ('stay', 5383),
 ('cold', 5087),
 ('check', 5068),
 ('excel', 5066),
 ('one', 5047),
 ('comfort', 4815)]

In [22]:
print("========= HOTEL REVIEW REPORT =========")

print("\nTotal Reviews:", len(df))

print("\nAverage Rating:", round(df["Reviewer_Score"].mean(),2))

print("\nSentiment Distribution:")

print(df["Sentiment"].value_counts())

print("\nTop Complaint Words:")

print(common.most_common(10))

========= HOTEL REVIEW REPORT =========

Total Reviews: 515738

Average Rating: 8.4

Sentiment Distribution:
Sentiment
Positive    362137
Neutral     101623
Negative     51978
Name: count, dtype: int64

Top Complaint Words:
[('room', 46627), ('staff', 19673), ('small', 19171), ('hotel', 18068), ('locat', 17632), ('breakfast', 12362), ('bed', 11453), ('posit', 11089), ('poor', 8287), ('bad', 6435)]
